In [72]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

In [42]:
order = pd.read_csv('/Users/yanlingliu/NUS/intern_competition/MEITUAN/①Keemat业务选题_order_info_脱敏数据表.csv')
exposure = pd.read_csv('/Users/yanlingliu/NUS/intern_competition/MEITUAN/②Keemart业务选题_exposure_info_脱敏数据表.csv')
activity = pd.read_csv('/Users/yanlingliu/NUS/intern_competition/MEITUAN/③Keemart业务选题_activity_timeline_脱敏数据表.csv')
order.sort_values(['dt', 'base_sku_id'], inplace=True)

In [40]:
exposure.head()

,dt,base_sku_id,buy_uv,view_uv
0,20251129,SKU599,5583,1628429
1,20251129,SKU158,5501,1556153
2,20251101,SKU599,4187,1545565
3,20251024,SKU610,4187,1529842
4,20251025,SKU1258,9606,1522995


In [41]:
activity.head()

,日期,营销活动
0,2025/8/25,8月活动
1,2025/8/26,8月活动
2,2025/8/27,8月活动
3,2025/8/28,8月活动
4,2025/8/29,8月活动


In [36]:
order.head()

,dt,stat_pay_user_id,stat_pay_main_order_id,category_name_cn,base_sku_name_en,brand_name_en_new,base_sku_id,sku_sale_amt,sku_sale_num,biz_total_discount_amt
72,20250901,User3625,Order18941,苏打汽水,brand143 red can sparkling drink (chilled) 240ml,Brand143,SKU1003,3.675,1,0.075
74,20250901,User3797,Order19320,苏打汽水,brand143 red can sparkling drink (chilled) 240ml,Brand143,SKU1003,3.750,1,0.000
175,20250901,User10085,Order50373,苏打汽水,brand143 red can sparkling drink (chilled) 240ml,Brand143,SKU1003,2.790,1,0.075
182,20250901,User10210,Order54077,苏打汽水,brand143 red can sparkling drink (chilled) 240ml,Brand143,SKU1003,3.675,1,0.075
228,20250901,User13803,Order70064,苏打汽水,brand143 red can sparkling drink (chilled) 240ml,Brand143,SKU1003,3.675,1,0.075


In [48]:
# 转成 datetime
activity['日期'] = pd.to_datetime(activity['日期'])
order['dt'] = pd.to_datetime(order['dt'].astype(str), format='%Y%m%d')

# 再匹配
order['activity'] = order['dt'].isin(activity['日期']).astype(int)

In [50]:
order.head()

,dt,stat_pay_user_id,stat_pay_main_order_id,category_name_cn,base_sku_name_en,brand_name_en_new,base_sku_id,sku_sale_amt,sku_sale_num,biz_total_discount_amt,activity
72,2025-09-01,User3625,Order18941,苏打汽水,brand143 red can sparkling drink (chilled) 240ml,Brand143,SKU1003,3.675,1,0.075,1
74,2025-09-01,User3797,Order19320,苏打汽水,brand143 red can sparkling drink (chilled) 240ml,Brand143,SKU1003,3.750,1,0.000,1
175,2025-09-01,User10085,Order50373,苏打汽水,brand143 red can sparkling drink (chilled) 240ml,Brand143,SKU1003,2.790,1,0.075,1
182,2025-09-01,User10210,Order54077,苏打汽水,brand143 red can sparkling drink (chilled) 240ml,Brand143,SKU1003,3.675,1,0.075,1
228,2025-09-01,User13803,Order70064,苏打汽水,brand143 red can sparkling drink (chilled) 240ml,Brand143,SKU1003,3.675,1,0.075,1


### 针对曝光率分析发现活动前后曝光率无明显提升

In [52]:
exposure['CVR'] = exposure['buy_uv'] / exposure['view_uv']

In [54]:
exposure['dt'] = pd.to_datetime(exposure['dt'].astype(str), format='%Y%m%d')
exposure['activity'] = exposure['dt'].isin(activity['日期']).astype(int)

In [58]:
exposure['day'] = pd.to_datetime(exposure['dt']).dt.day
exposure['dow'] = pd.to_datetime(exposure['dt']).dt.weekday

In [64]:
exposuremodel = smf.ols(
    'CVR ~ activity + C(base_sku_id) + C(day) + C(dow)',
    data=exposure
).fit()

In [66]:
print(exposuremodel.params['activity'])
print(exposuremodel.pvalues['activity'])

0.00021958968264608177
0.4163248947024871


In [67]:
exposuremodel1 = smf.ols(
    'view_uv ~ activity + C(base_sku_id) + C(day) + C(dow)',
    data=exposure
).fit()

In [68]:
print(exposuremodel1.params['activity'])
print(exposuremodel1.pvalues['activity'])

-3710.7415658000095
0.0012565253593491991


### 看order表

In [71]:
exposure

,dt,base_sku_id,buy_uv,view_uv,CVR,activity,day,dow
0,2025-11-29,SKU599,5583,1628429,0.003428,1,29,5
1,2025-11-29,SKU158,5501,1556153,0.003535,1,29,5
2,2025-11-01,SKU599,4187,1545565,0.002709,1,1,5
3,2025-10-24,SKU610,4187,1529842,0.002737,1,24,4
4,2025-10-25,SKU1258,9606,1522995,0.006307,1,25,5
...,...,...,...,...,...,...,...,...
100685,2025-11-23,SKU5224,82,190,0.431579,1,23,6
100686,2025-10-01,SKU5710,82,190,0.431579,1,1,2
100687,2025-10-30,SKU5592,82,127,0.645669,1,30,3
100688,2025-11-04,SKU1384,82,63,1.301587,0,4,1


In [73]:
# 1️⃣ baseline：非活动期的平均曝光（反事实）
baseline = exposure[exposure['activity']==0].groupby('base_sku_id')['view_uv'].mean().reset_index()
baseline.rename(columns={'view_uv':'baseline_view_uv'}, inplace=True)

df = exposure.merge(baseline, on='base_sku_id', how='left')

# 2️⃣ 只保留活动期数据来计算曝光增量
df_act = df[df['activity']==1].copy()

# 避免除0
df_act = df_act[df_act['baseline_view_uv'] > 0]

# 3️⃣ 曝光增量（只在活动期算）
df_act['exposure_increase'] = (
    df_act['view_uv'] - df_act['baseline_view_uv']
) / df_act['baseline_view_uv']

# 4️⃣ SKU级别聚合（活动期平均曝光增量）
sku_exposure = df_act.groupby('base_sku_id')['exposure_increase'].mean()

# 5️⃣ 选对照组（曝光几乎没涨的SKU）
threshold = 0.1   # 可以调，比如 5% / 10%
control_skus = sku_exposure[sku_exposure < threshold].index.tolist()

# 标记组别
df_act['group'] = np.where(df_act['base_sku_id'].isin(control_skus),
                          'control', 'treatment')

In [80]:
df_act[['base_sku_id','group']]

,base_sku_id,group
0,SKU599,treatment
1,SKU158,treatment
2,SKU599,treatment
3,SKU610,treatment
4,SKU1258,treatment
...,...,...
100653,SKU5756,control
100661,SKU5169,treatment
100668,SKU4529,control
100669,SKU5676,control
